# 07 - Analyses bonus sur le modèle général retenu

1. **Lissage temporel des prédictions** : exploiter le fait qu'un comportement dure plusieurs secondes ;
2. **UMAP des erreurs** : où se situent les erreurs dans l'espace des variables ?
3. **Prédiction conforme** : quantifier l'incertitude par des ensembles de prédiction à couverture garantie.

Chaque analyse respecte la règle du projet : **aucun réglage n'est fait sur le test**.

In [ ]:
import sys
from pathlib import Path

# Racine du projet ajoutée au chemin d'import (le notebook est dans notebooks/)
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config as cfg
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

import json
from src import data, labels, modeling, evaluation, smoothing, conformal, viz

In [ ]:
windows = data.load_table(cfg.WINDOWS_FILE)
dogs = data.load_dogs()
train_ids, test_ids = data.load_split()
pointeur = json.loads((cfg.TABLES_DIR / "meilleur_modele_general.json").read_text())
general = modeling.load_bundle(**pointeur)
classes = list(general["label_encoder"].classes_)

# Même tâche et même filtrage qu'au notebook 04
task = labels.select_task_windows(windows, "general", min_purity=cfg.MIN_PURITY)
task = task.merge(dogs[["DogID"] + cfg.STATIC_FEATURES], on="DogID", how="left")
task = task[task["target"].isin(classes)]
train, test = task[task["DogID"].isin(train_ids)], task[task["DogID"].isin(test_ids)]
print(f"Modèle général : {general['model_name']} / {general['experiment']} | {len(classes)} classes")
print(f"Train : {len(train):,} fenêtres | Test : {len(test):,} fenêtres")

pred_test = data.load_table(cfg.PROCESSED_DIR / "predictions_test_general.parquet")
pred_test = pred_test.set_index("window_id").loc[test["window_id"]].reset_index()
proba_test = pred_test[[f"proba_{c}" for c in classes]].to_numpy()
le = general["label_encoder"]

## 1. Lissage temporel

On moyenne les probabilités de `k` fenêtres consécutives (fenêtre centrée), sans traverser de rupture.
Le choix de `k` se fait sur des **prédictions hors-pli du train** (chaque fenêtre prédite par un modèle qui
n'a pas vu son chien), jamais sur le test. Le lissage n'utilise que les prédictions : aucune fuite de label.
Limite : la fenêtre centrée utilise le « futur » ; en temps réel il faudrait un lissage causal.

In [ ]:
oof = modeling.oof_predict_proba(general, train)          # long : 5 réentraînements
y_train = le.transform(train["target"])
choix_k = smoothing.choose_k(oof, y_train, train)
display(choix_k.round(4))
k_opt = int(choix_k.loc[choix_k["f1_macro_oof_train"].idxmax(), "k"])
print(f"k retenu sur le train : {k_opt} fenêtres (~{choix_k.set_index('k').loc[k_opt, 'duree_lissage_s']} s)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(choix_k["k"], choix_k["f1_macro_oof_train"], marker="o")
ax.axvline(k_opt, color="red", ls="--"); ax.set_xlabel("k (nombre de fenêtres lissées)")
ax.set_ylabel("F1 macro hors-pli (train)"); ax.set_title("Choix de la taille du lissage")
plt.tight_layout(); viz.savefig(fig, "07_choix_lissage"); plt.show()

In [ ]:
proba_lisse = smoothing.smooth_proba(proba_test, pred_test, k_opt)
pred_test["pred_lisse"] = le.inverse_transform(proba_lisse.argmax(axis=1))
avant = evaluation.compute_metrics(pred_test["target"], pred_test["pred"], labels=classes)
apres = evaluation.compute_metrics(pred_test["target"], pred_test["pred_lisse"], labels=classes)
display(pd.DataFrame({"sans lissage": avant, f"lissage k={k_opt}": apres}).round(3))

pc_avant = evaluation.per_dog_scores(pred_test, "pred")
pc_apres = evaluation.per_dog_scores(pred_test, "pred_lisse")
comp = pc_avant[["DogID", "f1_macro"]].merge(pc_apres[["DogID", "f1_macro"]], on="DogID",
                                             suffixes=("_avant", "_apres"))
print(f"Chiens améliorés par le lissage : {(comp['f1_macro_apres'] > comp['f1_macro_avant']).sum()} / {len(comp)}")
display(comp.round(3))

In [ ]:
d, t = pred_test[["DogID", "TestNum"]].iloc[0]
seance = pred_test[(pred_test["DogID"] == d) & (pred_test["TestNum"] == t)]
viz.plot_timeline(seance, pred_col="pred", show=("behavior_1",), title=f"Chien {d} - sans lissage").show()
viz.plot_timeline(seance, pred_col="pred_lisse", show=("behavior_1",), title=f"Chien {d} - avec lissage k={k_opt}").show()

## 2. UMAP des erreurs

Projection 2D des fenêtres de test (variables retenues par le modèle, standardisées). Questions :
- les erreurs sont-elles **aux frontières** entre nuages de classes (confusions intrinsèques, ex. marche/trot) ?
- ou **au cœur** d'un nuage d'une autre classe (label douteux, transition mal annotée) ?
- les fenêtres se regroupent-elles **par chien** plutôt que par comportement (forte variabilité individuelle) ?

UMAP est une méthode de visualisation : les distances dans le plan ne sont qu'indicatives.

In [ ]:
MAX_POINTS = 20000
vis = pred_test.sample(n=min(MAX_POINTS, len(pred_test)), random_state=cfg.RANDOM_STATE)
retenues = modeling.selected_features(general["model"], general["features"])
X_vis = test.set_index("window_id").loc[vis["window_id"], retenues].to_numpy()
emb = viz.umap_embedding(X_vis)
fig = viz.plot_umap_errors(emb, vis["target"], vis["pred"], dog_ids=vis["DogID"])
viz.savefig(fig, "07_umap_erreurs"); plt.show()

## 3. Prédiction conforme

Le modèle renvoie un **ensemble** de comportements, construit pour contenir le vrai comportement avec une
probabilité d'au moins 1 - α. Un ensemble d'une seule classe signale une prédiction sûre ; un ensemble de
plusieurs classes signale une ambiguïté (souvent entre classes réellement proches).

Protocole :
1. on sépare les chiens du train en **apprentissage** (75 %) et **calibration** (25 %) ;
2. le modèle retenu est réentraîné (mêmes hyperparamètres) sur les chiens d'apprentissage ;
3. le seuil est calibré sur les chiens de calibration ;
4. couverture et taille des ensembles sont mesurées sur les chiens de **test**.

Hypothèse à discuter : la garantie suppose des exemples échangeables. Les fenêtres d'un même chien sont
corrélées ; la couverture peut donc varier d'un chien à l'autre (mesurée plus bas).

In [ ]:
ALPHA = 0.10
fit_df, cal_df = conformal.calibration_split(train)
# une classe absente de l'apprentissage rendrait le modèle inutilisable : on vérifie
assert set(fit_df["target"]) == set(classes), "Classe absente des chiens d'apprentissage : changer la graine."
modele_conf = modeling.fit_clone(general, fit_df)
X_cal, y_cal = cal_df[general["features"]], le.transform(cal_df["target"])
X_te, y_te = test[general["features"]], le.transform(test["target"])
print(f"Apprentissage : {fit_df['DogID'].nunique()} chiens | calibration : {cal_df['DogID'].nunique()} chiens")

# Implémentation maison (LAC)
qhat = conformal.lac_threshold(modele_conf.predict_proba(X_cal), y_cal, ALPHA)
sets_maison = conformal.lac_sets(modele_conf.predict_proba(X_te), qhat)
resultats = {"maison_LAC": conformal.set_metrics(sets_maison, y_te)}

# MAPIE (LAC et APS)
try:
    for methode in ["lac", "aps"]:
        s = conformal.mapie_sets(modele_conf, X_cal, y_cal, X_te, ALPHA, method=methode)
        resultats[f"MAPIE_{methode.upper()}"] = conformal.set_metrics(s, y_te)
except Exception as e:
    print("MAPIE indisponible ou API différente :", repr(e))
display(pd.DataFrame(resultats).T.round(3))
print(f"Couverture visée : {1 - ALPHA:.0%}")

In [ ]:
par_classe = conformal.coverage_by(sets_maison, y_te, le.inverse_transform(y_te), "comportement")
par_chien = conformal.coverage_by(sets_maison, y_te, test["DogID"].to_numpy(), "DogID")
display(par_classe)
display(par_chien)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sizes = sets_maison.sum(axis=1)
axes[0].bar(*np.unique(sizes, return_counts=True), color="#5b8db8")
axes[0].set_xlabel("Taille de l'ensemble de prédiction"); axes[0].set_ylabel("Nb de fenêtres de test")
axes[0].set_title(f"Distribution des tailles (alpha = {ALPHA})")
axes[1].bar(par_chien["DogID"].astype(str), par_chien["couverture"], color="#9ecae1")
axes[1].axhline(1 - ALPHA, color="red", ls="--", label="couverture visée")
axes[1].set_ylim(0, 1.05); axes[1].set_xlabel("Chien de test"); axes[1].set_title("Couverture par chien")
axes[1].legend()
plt.tight_layout(); viz.savefig(fig, "07_conforme"); plt.show()

In [ ]:
# Compromis couverture / taille selon alpha (implémentation maison)
p_cal, p_te = modele_conf.predict_proba(X_cal), modele_conf.predict_proba(X_te)
courbe = []
for a in [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]:
    s = conformal.lac_sets(p_te, conformal.lac_threshold(p_cal, y_cal, a))
    courbe.append({"alpha": a, "couverture_visee": 1 - a, **conformal.set_metrics(s, y_te)})
courbe = pd.DataFrame(courbe)
display(courbe.round(3))
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(courbe["couverture_visee"], courbe["couverture"], marker="o", label="couverture obtenue")
ax.plot([0.7, 1], [0.7, 1], "k--", lw=1, label="idéal")
ax2 = ax.twinx(); ax2.plot(courbe["couverture_visee"], courbe["taille_moyenne"], color="orange", marker="s")
ax2.set_ylabel("taille moyenne des ensembles", color="orange"); ax2.grid(False)
ax.set_xlabel("couverture visée (1 - alpha)"); ax.set_ylabel("couverture obtenue (test)"); ax.legend(loc="upper left")
plt.tight_layout(); viz.savefig(fig, "07_courbe_conforme"); plt.show()

**Éléments de discussion** :
- la couverture obtenue sur le test est-elle proche de la couverture visée ? Un écart peut venir de la
  non-échangeabilité (chiens différents entre calibration et test) ;
- les classes avec de grands ensembles sont celles que le modèle confond (à rapprocher de la matrice de
  confusion) ;
- LAC donne les plus petits ensembles en moyenne ; APS adapte mieux la taille à la difficulté de chaque
  fenêtre ;
- intérêt pratique : un système réel pourrait ne répondre que lorsque l'ensemble est un singleton.
- avec LAC, un ensemble peut être **vide** quand aucune classe n'est assez probable : c'est un signal
  « je ne sais pas » (fenêtre atypique), à distinguer d'une erreur.